# Euclid DR3: 2-pt vs Stage-II posteriors

Compares the **2-pt (bandpower-only)** posteriors against the **Stage-II hybrid**
(bandpowers + field-level convergence maps) posteriors on the Euclid DR3 GLASS mocks,
with particular attention to the dark-energy equation of state $w$.

**Dependencies: `numpy` and `matplotlib` only.** Corner plots, credible contours and the
paper colour scheme are all implemented inline.

**Data:** a single `euclid_posterior_bundle.npz` holding full 10 000-draw posteriors for a
subsample of cosmologies (`float16`, plotting precision) plus `float64` per-cosmology
summaries over the *entire* test set, so the aggregate statistics use every test cosmology.

> Each arm ran as two repeats (`r0`, `r1`). All four runs share the **same 2041-cosmology
> test set in the same order** (verified from the eval output), so every comparison here is
> paired per cosmology, and `r0` vs `r1` differ only by training initialisation.

**Which checkpoints.** The Stage-II arm uses the *image-matched* LR schedule (155 epochs /
37 500-step cycles) **restarted once from its own finished weights**, i.e. 310 cumulative
epochs of training — experiment `euclid_hybrid_z8_resnet_imgmatch_resume`. The summary cell
below prints the exact experiment and run folder behind every column, so the notebook always
says which models produced the numbers it is showing.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

BUNDLE = 'euclid_posterior_bundle.npz'


## Style
Reproduces the `scienceplots` `['science', 'muted']` look used for the paper figures
(Tol *muted* palette, serif, inward ticks, 18 pt) without requiring `scienceplots`.
`usetex` is off by default so this runs without a LaTeX install.

In [ ]:

# Tol "muted" cycle, read out of scienceplots' `muted` style (see plot_common.muted_colors).
MUTED = ['#CC6677', '#332288', '#DDCC77', '#117733', '#88CCEE',
         '#882255', '#44AA99', '#999933', '#AA4499', '#DDDDDD']

# Paper convention (plot_common.py): w0 is labelled `w`, NOT `w_0`.
PARAM_LABELS = {"omega_m": r"$\Omega_\mathrm{m}$", "sigma_8": r"$\sigma_8$", "w0": r"$w$"}

C_BAND, C_HYB = MUTED[1], MUTED[0]   # indigo = 2-pt, rose = Stage II


def use_paper_style(font_size=18, usetex=False):
    """scienceplots['science','muted'] reproduced with plain matplotlib rcParams.

    usetex defaults OFF so the notebook runs without a LaTeX install; the paper figures
    used usetex=True. mathtext renders these labels essentially identically.
    """
    plt.rcParams.update({
        "font.family": "serif", "mathtext.fontset": "dejavuserif",
        "text.usetex": usetex, "font.size": font_size,
        "axes.prop_cycle": plt.cycler("color", MUTED),
        "axes.linewidth": 0.5, "axes.grid": False,
        "lines.linewidth": 1.5,
        "xtick.direction": "in", "ytick.direction": "in",
        "xtick.top": True, "ytick.right": True,
        "xtick.major.width": 0.5, "ytick.major.width": 0.5,
        "xtick.minor.visible": True, "ytick.minor.visible": True,
        "legend.frameon": False,
        "figure.dpi": 120, "savefig.bbox": "tight",
    })


def _gauss_kernel(sigma, truncate=4.0):
    r = max(int(truncate * sigma + 0.5), 1)
    x = np.arange(-r, r + 1, dtype=float)
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum()


def _smooth2d(a, sigma):
    """Separable Gaussian blur, numpy only (stand-in for scipy.ndimage.gaussian_filter)."""
    if sigma <= 0:
        return a
    k = _gauss_kernel(sigma)
    pad = len(k) // 2
    out = np.apply_along_axis(
        lambda m: np.convolve(np.pad(m, pad, mode="edge"), k, mode="valid"), 0, a)
    out = np.apply_along_axis(
        lambda m: np.convolve(np.pad(m, pad, mode="edge"), k, mode="valid"), 1, out)
    return out


def hpd_levels(h, probs=(0.68, 0.95)):
    """Density levels enclosing the requested posterior mass (highest-density first)."""
    flat = np.sort(h.ravel())[::-1]
    csum = np.cumsum(flat)
    csum /= csum[-1]
    lv = [flat[np.searchsorted(csum, p)] for p in probs]
    return sorted(set(lv))


def contour2d(ax, x, y, color, bins=64, sigma=1.4, fill=True, lw=1.4, label=None, zorder=2):
    """68/95% credible contours from raw draws, via a smoothed 2-D histogram."""
    h, xe, ye = np.histogram2d(x, y, bins=bins)
    h = _smooth2d(h, sigma)
    xc, yc = 0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1])
    levels = hpd_levels(h)
    if fill:
        ax.contourf(xc, yc, h.T, levels=levels + [h.max() * 1.01],
                    colors=[color, color], alpha=0.25, zorder=zorder)
    ax.contour(xc, yc, h.T, levels=levels, colors=color, linewidths=lw, zorder=zorder + 1)
    if label:
        ax.plot([], [], color=color, lw=lw, label=label)


def hist1d(ax, v, color, bins=60, lw=1.5, label=None, sigma=1.2):
    h, e = np.histogram(v, bins=bins, density=True)
    if sigma > 0:                      # same Gaussian smoothing as the 2-D panels
        k = _gauss_kernel(sigma); pad = len(k) // 2
        h = np.convolve(np.pad(h, pad, mode="edge"), k, mode="valid")
    c = 0.5 * (e[1:] + e[:-1])
    ax.plot(c, h / h.max(), color=color, lw=lw, label=label)
    ax.fill_between(c, 0, h / h.max(), color=color, alpha=0.18)


def corner(samples, labels, truth=None, colors=None, names=None, figsize=None, bins=64):
    """Minimal corner plot overlaying several posteriors on one set of axes.

    samples : list of (n_draws, n_dim) arrays
    """
    d = len(labels)
    figsize = figsize or (2.6 * d, 2.6 * d)
    fig, axes = plt.subplots(d, d, figsize=figsize)
    colors = colors or MUTED
    for i in range(d):
        for j in range(d):
            ax = axes[i, j]
            if j > i:
                ax.axis("off")
                continue
            for k, s in enumerate(samples):
                if i == j:
                    hist1d(ax, s[:, i], colors[k], bins=bins,
                           label=(names[k] if names else None))
                else:
                    contour2d(ax, s[:, j], s[:, i], colors[k], bins=bins)
            if truth is not None:
                if i == j:
                    ax.axvline(truth[i], color="0.25", lw=1.0, ls="--")
                else:
                    ax.axvline(truth[j], color="0.25", lw=1.0, ls="--")
                    ax.axhline(truth[i], color="0.25", lw=1.0, ls="--")
            if i == d - 1:
                ax.set_xlabel(labels[j])
            else:
                ax.set_xticklabels([])
            if j == 0 and i > 0:
                ax.set_ylabel(labels[i])
            elif i == j:
                ax.set_yticks([])
            else:
                ax.set_yticklabels([])
    fig.subplots_adjust(wspace=0.08, hspace=0.08)
    return fig, axes


In [ ]:
use_paper_style()

d = np.load(BUNDLE, allow_pickle=False)
meta = json.loads(str(d['meta']))
PARAMS = meta['params']
LABELS = [meta['param_labels'][p] for p in PARAMS]
REPEATS = meta.get('repeats', [0, 1])

def show_ids(repeat=0):
    """Plotted cosmologies for a repeat, most-realistic-first."""
    key = 'show_sim_ids_r%d' % repeat
    if key not in d.files:          # tolerate a pre-per-repeat bundle
        return d['show_sim_ids']
    return d[key]

print('parameters :', PARAMS)
for k, v in meta['runs'].items():
    print('  ' + k.ljust(10) + v['label'].ljust(24)
          + 'N_test=' + str(v['n_test']).rjust(5) + '  draws=' + str(v['n_draws']))
    print(' ' * 13 + v['experiment'] + '/' + v.get('run_folder', '?'))

for r in REPEATS:
    s8 = d['show_S8_r%d' % r]
    ib = d['show_in_fiducial_box_r%d' % r]
    print('repeat %d: %d plotted cosmologies, %d inside the fiducial box, '
          'S8 %.3f-%.3f' % (r, len(s8), int(ib.sum()), s8.min(), s8.max()))


## 1. Posterior comparison for individual cosmologies
Both arms see the *same* mock; the only difference is whether the field-level maps are used.

`show_ids(repeat)` (stored as `show_sim_ids_r<repeat>`) is ordered
**most-realistic-first** (closest to $S_8=0.80$, $w=-1$,
$\Omega_\mathrm{m}=0.30$), so the examples below sit near the fiducial cosmology
rather than out at the edges of the broad Gower prior. Index further into `SHOW` to
see progressively more extreme cosmologies.


In [ ]:
def corner_for(sim_id, repeat=0, bins=64):
    SHOW = show_ids(repeat)
    i = int(np.where(SHOW == sim_id)[0][0])
    b = d['band_r%d__samples' % repeat][i].astype(np.float64)
    h = d['hybrid_r%d__samples' % repeat][i].astype(np.float64)
    truth = d['band_r%d__theta0s' % repeat][i]
    fig, axes = corner([b, h], LABELS, truth=truth, colors=[C_BAND, C_HYB],
                       names=['2-pt (bandpowers)', 'Stage II (hybrid)'], bins=bins)
    handles, lbls = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, lbls, loc='upper right', bbox_to_anchor=(0.98, 0.98))
    s8 = truth[PARAMS.index('sigma_8')] * np.sqrt(truth[PARAMS.index('omega_m')] / 0.3)
    fig.suptitle('sim %d  (repeat %d)   $S_8$=%.3f, $w$=%.2f, $\\Omega_m$=%.3f'
                 % (sim_id, repeat, s8, truth[PARAMS.index('w0')],
                    truth[PARAMS.index('omega_m')]), x=0.62, y=1.01)
    return fig


fig = corner_for(show_ids(0)[0])


In [ ]:
# a few more, to show the improvement is not cherry-picked
for sid in show_ids(0)[1:4]:
    corner_for(sid)


## 2. Constraining power on $w$
68 % credible width per cosmology, over the **full** test set.

In [ ]:
def paired(param, repeat):
    """Match band and hybrid cosmology-by-cosmology (they share a split per repeat)."""
    bi = d['band_r%d__simids_all' % repeat]
    hi = d['hybrid_r%d__simids_all' % repeat]
    common = np.intersect1d(bi, hi)
    bpos = {int(v): k for k, v in enumerate(bi)}
    hpos = {int(v): k for k, v in enumerate(hi)}
    j = PARAMS.index(param)
    wb = d['band_r%d__w68_all' % repeat][[bpos[int(v)] for v in common], j]
    wh = d['hybrid_r%d__w68_all' % repeat][[hpos[int(v)] for v in common], j]
    return common, wb, wh

for rep in (0, 1):
    _, wb, wh = paired('w0', rep)
    r = np.median(wh / wb)
    print('repeat %d: median w-width  2-pt %.4f -> Stage II %.4f   ratio %.3f  '
          '(%.1f%% tighter, N=%d)' % (rep, np.median(wb), np.median(wh), r,
                                      100 * (1 - r), len(wb)))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
common, wb, wh = paired('w0', 0)

bins = np.linspace(0, max(wb.max(), wh.max()) * 1.02, 45)
axes[0].hist(wb, bins=bins, color=C_BAND, alpha=0.55, label='2-pt (bandpowers)')
axes[0].hist(wh, bins=bins, color=C_HYB, alpha=0.55, label='Stage II (hybrid)')
axes[0].set_xlabel('68% credible width on $w$')
axes[0].set_ylabel('cosmologies')
axes[0].legend()

axes[1].scatter(wb, wh, s=9, color=C_HYB, alpha=0.55, edgecolors='none')
lim = [0, max(wb.max(), wh.max()) * 1.05]
axes[1].plot(lim, lim, color='0.35', lw=1.0, ls='--', label='no improvement')
axes[1].set_xlim(lim)
axes[1].set_ylim(lim)
axes[1].set_xlabel('2-pt 68% width on $w$')
axes[1].set_ylabel('Stage II 68% width on $w$')
axes[1].legend()
fig.tight_layout()


### All three parameters

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
x = np.arange(len(PARAMS))
for rep, mk in zip((0, 1), ('o-', 's-')):
    ratios = [np.median(paired(p, rep)[2] / paired(p, rep)[1]) for p in PARAMS]
    ax.plot(x, ratios, mk, color=MUTED[rep], label='repeat %d' % rep, ms=9)
ax.axhline(1.0, color='0.35', lw=1.0, ls='--')
ax.set_xticks(x)
ax.set_xticklabels(LABELS)
ax.set_ylabel('median width ratio\n(Stage II / 2-pt)')
ax.set_ylim(0, 1.25)
ax.legend()
fig.tight_layout()


## 3. Shrinkage: every parameter, both arms

**Shrinkage** = median posterior 68 % width / prior 68 % width. The prior width is measured
from the spread of the *true* parameters across the test set, which is itself a draw from the
Gower prior — so shrinkage 0.25 means the posterior is 4x tighter than the prior.
The last column is what we actually care about: how much the field-level maps add on top of
the 2-pt information.


In [ ]:
ROWS = PARAMS + ['S8']

def post_width(arm, rep, param):
    if param == 'S8':
        return d['%s_r%d__s8_w68_all' % (arm, rep)]
    return d['%s_r%d__w68_all' % (arm, rep)][:, PARAMS.index(param)]

def prior_width(arm, rep, param):
    """68% spread of the TRUE parameters over the test set = a prior sample."""
    if param == 'S8':
        th = d['%s_r%d__s8_true_all' % (arm, rep)]
    else:
        th = d['%s_r%d__theta_all' % (arm, rep)][:, PARAMS.index(param)]
    q16, q84 = np.quantile(th, [0.16, 0.84])
    return q84 - q16

hdr = ('param'.ljust(10) + 'rep'.rjust(4) + 'prior68'.rjust(10) + '2-pt w68'.rjust(10)
       + 'SII w68'.rjust(10) + '2-pt shrink'.rjust(13) + 'SII shrink'.rjust(12)
       + 'SII/2-pt'.rjust(10))
print(hdr)
print('-' * len(hdr))
summary = {}
for param in ROWS:
    for rep in REPEATS:
        pw = prior_width('band', rep, param)
        wb = np.median(post_width('band', rep, param))
        wh = np.median(post_width('hybrid', rep, param))
        summary[(param, rep)] = (pw, wb, wh)
        print(param.ljust(10) + str(rep).rjust(4)
              + ('%.4f' % pw).rjust(10) + ('%.4f' % wb).rjust(10) + ('%.4f' % wh).rjust(10)
              + ('%.3f' % (wb / pw)).rjust(13) + ('%.3f' % (wh / pw)).rjust(12)
              + ('%.3f' % (wh / wb)).rjust(10))
    print()


In [ ]:
# paired per-cosmology improvement, all parameters at once
fig, axes = plt.subplots(1, len(ROWS), figsize=(3.2 * len(ROWS), 3.6), sharey=True)
for ax, param in zip(np.atleast_1d(axes), ROWS):
    for rep, col in zip(REPEATS, MUTED):
        wb, wh = post_width('band', rep, param), post_width('hybrid', rep, param)
        n = min(len(wb), len(wh))
        ax.hist(wh[:n] / wb[:n], bins=np.linspace(0, 1.4, 40), color=col,
                alpha=0.5, label='repeat %d' % rep)
    ax.axvline(1.0, color='0.35', lw=1.0, ls='--')
    ax.set_xlabel(meta['param_labels'].get(param, '$S_8$'))
np.atleast_1d(axes)[0].set_ylabel('cosmologies')
np.atleast_1d(axes)[0].legend(fontsize=11)
fig.suptitle('68% width ratio, Stage II / 2-pt  (<1 = Stage II tighter)', y=1.04)
fig.tight_layout()


## 4. Figures of merit and information content

Every FoM the evaluation computes, all measured against the prior (higher = better):

| key | meaning |
|---|---|
| `fom` | full $d$-dimensional FoM over all inferred parameters |
| `fom_subsets.omega_m__sigma_8` | $(\Omega_\mathrm{m}, \sigma_8)$ |
| `fom_subsets.omega_m__w0` | $(\Omega_\mathrm{m}, w)$ |
| `fom_subsets.s8__w0` | $(S_8, w)$ — the dark-energy figure of merit |

`test_log_prob` is $\Delta$MI (information content, more negative = better);
`mahalanobis_distance_mean` should sit near $\sqrt{d}$ if the posteriors are unbiased and
correctly sized.


In [ ]:
FOM_KEYS = [('fom', 'FoM (all params)'),
            ('fom_subsets.omega_m__sigma_8', 'FoM (Om, s8)'),
            ('fom_subsets.omega_m__w0', 'FoM (Om, w)'),
            ('fom_subsets.s8__w0', 'FoM (S8, w)')]
EXTRA = [('test_log_prob', 'dMI [nats]'),
         ('sample_ensemble_mse', 'posterior-mean MSE'),
         ('mahalanobis_distance_mean', 'Mahalanobis mean')]

def dig(m, dotted):
    cur = m
    for part in dotted.split('.'):
        if not isinstance(cur, dict) or part not in cur:
            return None
        cur = cur[part]
    return cur if isinstance(cur, (int, float)) else None

M = {k: (v.get('metrics', {}) or {}) for k, v in meta['runs'].items()}
cols = list(M)
hdr = 'metric'.ljust(24) + ''.join(c.rjust(13) for c in cols) + 'SII/2-pt r0'.rjust(14)
print(hdr)
print('-' * len(hdr))
for key, label in FOM_KEYS + EXTRA:
    vals = [dig(M[c], key) for c in cols]
    line = label.ljust(24) + ''.join(('--' if v is None else '%.4g' % v).rjust(13) for v in vals)
    b, h = dig(M.get('band_r0', {}), key), dig(M.get('hybrid_r0', {}), key)
    line += ('--' if (b in (None, 0) or h is None) else '%.3f' % (h / b)).rjust(14)
    print(line)

print()
print('per-parameter width_68 / width_95 / bias as recorded by the evaluation:')
sub = 'param'.ljust(10) + ''.join((c + ' w68').rjust(16) for c in cols)
print(sub)
print('-' * len(sub))
for param in PARAMS + ['s8']:
    vals = [dig(M[c], param + '.width_68') for c in cols]
    print(param.ljust(10) + ''.join(('--' if v is None else '%.4g' % v).rjust(16) for v in vals))


## Caveats
* Two repeats per arm. Seed sensitivity was the main worry early on — the *step*-matched
  Stage-II campaign showed ~0.5 nats of seed-to-seed scatter — but the image-matched schedule
  largely removed it, and the extra 155 resumed epochs removed what was left: the two Stage-II
  repeats here agree to **0.007 nats** of $\Delta$MI (-6.677 vs -6.670) and ~0.4 % of FoM
  (812 vs 816). The per-cosmology *pairing* below is still the more robust comparison.
* Plotted draws are `float16`; all quoted widths and ratios come from the `float64` summaries.
* `w` is the constant dark-energy equation of state (`w0` in the configs), labelled $w$ to
  match the paper figures.
* The Gower prior only extends to $w = -1.0097$, so $w=-1$ sits essentially **at the
  lower prior edge**. Posteriors for near-fiducial cosmologies therefore pile up against
  that boundary, which compresses the apparent $w$ gain: the 68 % width improves only
  ~16-18 %, while the $(\Omega_\mathrm{m}, w)$ FoM — which also captures the reduced
  degeneracy — **more than triples** (11.6/11.1 $\to$ 38.1/38.2). Read the $w$ width
  improvement as a lower bound on what an uncapped prior would show.
